# 03 · Profit engine and segmentation

Milestone 3. Engine 2 turns each customer's propensity into money, and Engine 1 places each customer in one of four business segments.

The term deposit is a **funding product** (D-008). The bank doesn't earn the principal. It earns the funding spread (FTP rate minus customer rate) on the deposit over the term.

```
D_i  = clip(D_min + alpha x max(balance_i, 0), D_min, D_max)      deposit size
V_i  = D_i x spread x term - incentive - onboarding                 contribution if converted
C_i  = channel cost (fully loaded: mobile £12, landline £10, digital £2)
BE_i = (P_i - P0) x V_i                                             break-even contact cost (Q3)
EP_i = BE_i - C_i                                                   expected incremental profit
```

*All economic figures are illustrative assumptions, not bank performance. Balances are euros in the source data and are shown as £ for illustration.*

In [1]:
from dataclasses import asdict, replace
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pacopt import config as cfg, economics as econ, segmentation as seg, pipeline as pl, charts as ch

pio.renderers.default = 'notebook_connected'
pd.set_option('display.width', 140)
base = pl.load_universe().merge(pl.load_scored(), on='customer_id')   # features + propensity
t = pl.campaign_table()                                                # + economics + segments
print(t.shape)
a = asdict(cfg.ECONOMICS); a['spread'] = cfg.ECONOMICS.spread
pd.Series(a, name='default').to_frame()

(10000, 53)


,default
deposit_min,10000.0
deposit_alpha,1.0
deposit_max,100000.0
ftp_rate,0.0475
customer_rate,0.03
term_years,1.0
incentive,40.0
onboarding_cost,5.0
organic_rate,0.0
channel_cost,"{'mobile_call': 12.0, 'landline_call': 10.0, '..."


## 1. Calibrating the default assumptions (sub-task 3.3)
The plan's starting defaults (D_min £5,000, incentive £15) had two targets: (a) roughly 25-50% of eligible customers with positive EP, and (b) a **£50K budget that binds**, meaning the positive-EP customers, after channel capacity caps (calls 4,500, digital 2,500), cost more than £50K to contact.

In [2]:
scenarios = {
    'Plan start: D_min £5K, incentive £15': replace(cfg.ECONOMICS, deposit_min=5_000, incentive=15),
    'D_min £10K, incentive £15': replace(cfg.ECONOMICS, incentive=15),
    'D_min £10K, incentive £25': replace(cfg.ECONOMICS, incentive=25),
    'Chosen default: D_min £10K, incentive £40': cfg.ECONOMICS,
}
rows = {name: econ.budget_check(econ.add_economics(base, ec)) for name, ec in scenarios.items()}
calib = pd.DataFrame(rows).T[['positive_ep', 'positive_ep_share_of_eligible', 'cost_after_channel_caps',
                              'sum_positive_ep', 'budget_binds']]
calib

,positive_ep,positive_ep_share_of_eligible,cost_after_channel_caps,sum_positive_ep,budget_binds
"Plan start: D_min £5K, incentive £15",4739,0.524168,33070.0,50834.583007,False
"D_min £10K, incentive £15",7863,0.869705,58182.0,129182.708988,True
"D_min £10K, incentive £25",7433,0.822144,58170.0,118957.695195,True
"Chosen default: D_min £10K, incentive £40",6700,0.741068,54294.0,104414.218653,True


In [3]:
el = base[base['eligible'] == 1].assign(contact_cost=lambda d: econ.contact_cost(d['contact']),
                                        channel_group=lambda d: econ.channel_of(d['contact']).map(cfg.CHANNEL_GROUP).values)
ceiling = sum(g.nlargest(cfg.CHANNEL_CAPACITY[k], 'contact_cost')['contact_cost'].sum()
              for k, g in el.groupby('channel_group'))
print(f'Eligible customers: {len(el):,}  |  most the capped campaign could ever cost: £{ceiling:,.0f}')

Eligible customers: 9,041  |  most the capped campaign could ever cost: £59,000


**The two targets conflict.** Even if *every* eligible customer had positive EP, the channel caps limit the spend to about £59K. For the budget to bind, roughly 70% or more of eligible customers must have positive EP, which is above the 25-50% target. The plan sets the binding budget as the condition that makes the optimisation meaningful, so target (b) takes priority. The chosen defaults (D_min £10,000, incentive £40) give **74% positive EP and £54.3K of capped demand against £50K**. That is binding, but only by about 9%, and the incentive stays realistic (D-031). Every value is editable in the dashboard.

## 2. Per-customer economics (sub-task 3.4) and the break-even cost (Q3)

In [4]:
cols = ['deposit', 'value', 'contact_cost', 'break_even_cost', 'expected_profit', 'ep_per_cost']
t[cols].describe(percentiles=[.1, .5, .9, .99]).round(2)

,deposit,value,contact_cost,break_even_cost,expected_profit,ep_per_cost
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,11392.53,154.37,8.99,18.87,9.88,1.41
std,2969.60,51.97,4.47,24.28,23.26,2.33
min,10000.00,130.00,2.00,0.00,-8.90,-1.00
10%,10000.00,130.00,2.00,4.92,-3.60,-0.33
50%,10438.00,137.67,12.00,11.02,3.51,0.74
90%,13629.60,193.52,12.00,40.16,28.64,3.74
99%,23052.42,358.42,12.00,116.93,104.94,10.22
max,100000.00,1705.00,12.00,525.29,513.29,42.77


In [5]:
c = t.loc[t['expected_profit'].idxmax()]
e = cfg.ECONOMICS
print(f"Worked example: {c.customer_id}  balance £{c.balance:,.0f}  channel {c.channel}  P = {c.propensity:.3f}")
print(f"  D  = clip({e.deposit_min:,.0f} + {e.deposit_alpha} x {max(c.balance, 0):,.0f}) = £{c.deposit:,.0f}")
print(f"  V  = {c.deposit:,.0f} x {e.spread:.2%} x {e.term_years:g} - {e.incentive:g} - {e.onboarding_cost:g} = £{c.value:,.2f}")
print(f"  BE = {c.propensity:.3f} x {c.value:,.2f} = £{c.break_even_cost:,.2f}   (most worth spending to contact)")
print(f"  EP = {c.break_even_cost:,.2f} - {c.contact_cost:g} = £{c.expected_profit:,.2f}")

Worked example: C141694  balance £71,188  channel mobile_call  P = 0.382
  D  = clip(10,000 + 1.0 x 71,188) = £81,188
  V  = 81,188 x 1.75% x 1 - 40 - 5 = £1,375.79
  BE = 0.382 x 1,375.79 = £525.29   (most worth spending to contact)
  EP = 525.29 - 12 = £513.29


The break-even cost answers **Q3** directly: it is the most the bank should spend to contact a customer. Contact is worthwhile when BE_i is greater than the channel cost.

In [6]:
q3 = t.groupby('channel').agg(customers=('customer_id', 'size'), cost=('contact_cost', 'first'),
                              median_break_even=('break_even_cost', 'median'),
                              p90_break_even=('break_even_cost', lambda s: s.quantile(.9)),
                              share_worth_contacting=('positive_ep', 'mean'))
q3.round(2)

,customers,cost,median_break_even,p90_break_even,share_worth_contacting
channel,,,,,
digital,2881,2.0,6.15,10.46,0.96
landline_call,630,10.0,11.95,56.93,0.64
mobile_call,6489,12.0,14.09,50.41,0.60


## 3. Rankings: by EP and by return per £ (sub-task 3.5)
EP ranks customers by absolute profit. EP / C ranks them by profit per £ of budget, which is the right order when the budget is the binding limit. The greedy baseline (M4) uses it.

In [7]:
show = ['customer_id', 'segment', 'channel', 'propensity', 'value', 'contact_cost', 'expected_profit', 'ep_per_cost']
print('Top 8 by expected profit')
display(t.nsmallest(8, 'ep_rank')[show].round(3))
print('Top 8 by EP per £ of contact cost')
display(t.nsmallest(8, 'ep_per_cost_rank')[show].round(3))
k = 1_000
a_, b_ = set(t.nsmallest(k, 'ep_rank').customer_id), set(t.nsmallest(k, 'ep_per_cost_rank').customer_id)
print(f'Overlap of the two top-{k:,} lists: {len(a_ & b_) / k:.0%}  |  digital share: '
      f"by EP {t[t.customer_id.isin(a_)].channel.eq('digital').mean():.0%}, by EP/C {t[t.customer_id.isin(b_)].channel.eq('digital').mean():.0%}")

Top 8 by expected profit


,customer_id,segment,channel,propensity,value,contact_cost,expected_profit,ep_per_cost
9212,C141694,High value,mobile_call,0.382,1375.790,12.0,513.287,42.774
9839,C144522,High value,mobile_call,0.696,614.680,12.0,415.761,34.647
9547,C143291,High value,mobile_call,0.527,643.450,12.0,326.829,27.236
9141,C141383,High value,mobile_call,0.667,444.370,12.0,284.227,23.686
6966,C131223,High value,mobile_call,0.414,701.988,12.0,278.276,23.190
9414,C142718,High value,mobile_call,0.742,367.055,12.0,260.207,21.684
8951,C140555,High value,mobile_call,0.880,306.505,12.0,257.724,21.477
5847,C126228,High value,landline_call,0.150,1705.000,10.0,246.209,24.621


Top 8 by EP per £ of contact cost


,customer_id,segment,channel,propensity,value,contact_cost,expected_profit,ep_per_cost
9212,C141694,High value,mobile_call,0.382,1375.790,12.0,513.287,42.774
2045,C109422,High value,digital,0.143,515.315,2.0,71.617,35.808
9839,C144522,High value,mobile_call,0.696,614.680,12.0,415.761,34.647
9547,C143291,High value,mobile_call,0.527,643.450,12.0,326.829,27.236
2712,C112245,High value,digital,0.126,433.433,2.0,52.792,26.396
5847,C126228,High value,landline_call,0.150,1705.000,10.0,246.209,24.621
9575,C143424,High value,landline_call,0.900,275.320,10.0,237.788,23.779
9141,C141383,High value,mobile_call,0.667,444.370,12.0,284.227,23.686


Overlap of the two top-1,000 lists: 60%  |  digital share: by EP 1%, by EP/C 41%


## 4. Segmentation (sub-task 3.6): the Screen 1 view
A transparent propensity-by-value grid. Propensity is "high" in the top 40% of P_i, and value is "high" at or above the median of V_i. Every customer gets exactly one segment.

In [8]:
cut = seg.thresholds(t)
counts = t['segment'].value_counts().reindex(list(cfg.SEGMENTS))
fig = go.Figure(go.Bar(x=counts.index, y=counts.values, marker_color=ch.BAR_BLUE, marker_cornerradius=8,
                       hovertemplate='%{x}<br>%{y:,} customers<extra></extra>'))
ch.style(fig, 'Customers by segment', height=400, showlegend=False)
fig.update_yaxes(tickformat=',', rangemode='tozero')
fig.update_layout(bargap=0.35)
fig.show()
print(f"Cut-offs: propensity >= {cut['propensity']:.3f} | value >= £{cut['value']:.2f}   counts sum = {counts.sum():,}")

Cut-offs: propensity >= 0.084 | value >= £137.67   counts sum = 10,000


In [9]:
seg.segment_profile(t).round(3)

,customers,avg_age,avg_balance,avg_propensity,avg_value,avg_expected_profit,pct_eligible,pct_positive_ep,pct_never_contacted,avg_engagement,share
segment,,,,,,,,,,,
High value,2527,42.245,3136.454,0.217,184.830,28.573,0.989,0.997,0.696,15.587,0.253
Growing,1482,38.553,160.368,0.190,132.867,13.465,0.951,0.875,0.730,13.524,0.148
Dormant,2475,41.932,2179.928,0.059,168.149,3.543,0.996,0.728,0.884,4.365,0.248
Low engagement,3516,40.603,33.450,0.053,131.841,-0.602,0.759,0.405,0.886,4.409,0.352


In [10]:
colors = dict(zip(cfg.SEGMENTS, [ch.BLUE, ch.ORANGE, '#1baf7a', '#eda100']))
fig = go.Figure()
for s_ in cfg.SEGMENTS:
    d = t[t['segment'] == s_]
    fig.add_scattergl(x=d['propensity'], y=d['value'], mode='markers', name=f'{s_} ({len(d):,})',
                      marker=dict(color=colors[s_], size=5, opacity=0.55),
                      customdata=d[['customer_id', 'expected_profit']],
                      hovertemplate='%{customdata[0]}<br>P %{x:.1%}<br>V £%{y:,.0f}<br>EP £%{customdata[1]:,.2f}<extra>' + s_ + '</extra>')
fig.add_vline(x=cut['propensity'], line=dict(color=ch.MUTED, dash='dash', width=1.5))
fig.add_hline(y=cut['value'], line=dict(color=ch.MUTED, dash='dash', width=1.5))
ch.style(fig, 'Value by propensity: the four segment quadrants', height=480)
fig.update_xaxes(title='Propensity P_i', tickformat='.0%')
fig.update_yaxes(title='Contribution value V_i (£, log scale)', type='log')
fig.show()

**Overlays, not segments (D-033):** RFM scores from 1 to 5 (R from the recency bucket, F from `previous`, M from the balance quintile) and an engagement score from 0 to 100 (weights: recency 40%, frequency 30%, prior success 30%). The table shows each segment's averages.

In [11]:
ov = t.groupby('segment', observed=False)[['r_score', 'f_score', 'm_score', 'engagement_score']].mean()
ov['customers_75plus'] = t.groupby('segment', observed=False)['flag_age_75plus'].sum()
ov['ineligible'] = t.groupby('segment', observed=False)['eligible'].apply(lambda s: int((s == 0).sum()))
ov.round(2)

,r_score,f_score,m_score,engagement_score,customers_75plus,ineligible
segment,,,,,,
High value,1.76,1.71,4.30,15.59,42,29
Growing,1.62,1.65,2.04,13.52,16,73
Dormant,1.27,1.22,4.09,4.36,0,11
Low engagement,1.26,1.25,1.69,4.41,0,846


## 5. High-value opportunities (sub-task 3.7)

In [12]:
top = econ.top_decile_share(t)
print(f"Top EP decile: {top['customers']:,} eligible customers hold £{top['top_decile_ep']:,.0f} of "
      f"£{top['total_positive_ep']:,.0f} positive EP ({top['share']:.0%}); entry bar EP >= £{top['min_ep_in_top_decile']:.2f}")
dec = econ.ep_decile_table(t)
fig = go.Figure(go.Bar(x=dec.index.astype(str), y=dec['share_of_positive_ep'], marker_color=ch.BAR_BLUE,
                       marker_cornerradius=4, text=[f'{v:.0%}' if v >= 0.005 else '' for v in dec['share_of_positive_ep']],
                       textposition='outside', textfont=dict(color=ch.INK_2),
                       hovertemplate='EP decile %{x}<br>%{y:.1%} of positive EP<extra></extra>'))
ch.style(fig, 'Share of total positive expected profit by EP decile (1 = highest; eligible customers)',
         height=400, showlegend=False)
fig.update_xaxes(title='EP decile')
fig.update_yaxes(title='Share of positive EP', tickformat='.0%')
fig.show()
dec.round(3)

Top EP decile: 905 eligible customers hold £60,915 of £104,414 positive EP (58%); entry bar EP >= £31.30


,customers,mean_ep,positive_ep,mean_propensity,mean_value,mean_cost,share_of_positive_ep,cum_share
ep_decile,,,,,,,,
1,904,67.349,60883.378,0.428,205.546,11.715,0.583,0.583
2,904,20.229,18286.730,0.179,190.387,10.949,0.175,0.758
3,904,10.777,9742.653,0.132,165.206,9.615,0.093,0.852
4,904,6.901,6238.883,0.089,159.004,6.635,0.060,0.911
5,904,4.860,4393.734,0.064,148.130,4.624,0.042,0.953
6,904,3.363,3040.399,0.058,144.977,5.195,0.029,0.982
7,904,1.826,1650.531,0.058,141.276,6.569,0.016,0.998
8,904,-0.114,177.911,0.081,143.429,11.717,0.002,1.000
9,904,-1.563,0.000,0.076,135.613,11.865,0.000,1.000


In [13]:
t[t['eligible'] == 1].nsmallest(int(np.ceil((t['eligible'] == 1).sum() / 10)), 'ep_rank')['segment'].value_counts().reindex(list(cfg.SEGMENTS)).rename('customers in top EP decile').to_frame()

,customers in top EP decile
segment,
High value,697
Growing,205
Dormant,3
Low engagement,0


## 6. Sensitivity (sub-task 3.8)
Total positive EP across eligible customers when one driver changes by ±50% and everything else stays at the default. The spread changes through the FTP rate, with the customer rate held fixed.

In [14]:
sens = econ.sensitivity(base)
names = {'spread': 'Funding spread', 'incentive': 'Incentive', 'contact_cost': 'Contact cost'}
fig = go.Figure()
for (p, d), color in zip(sens.groupby('parameter', sort=False), [ch.BLUE, ch.ORANGE, '#1baf7a']):
    fig.add_scatter(x=d['factor'] - 1, y=d['total_positive_ep'], name=names[p], mode='lines+markers',
                    line=dict(color=color, width=2), marker=dict(size=8, line=dict(color=ch.SURFACE, width=2)),
                    hovertemplate=names[p] + ' %{x:+.0%}<br>positive EP £%{y:,.0f}<extra></extra>')
ch.style(fig, 'Total positive expected profit vs change in one driver', height=430)
fig.update_xaxes(title='Change from default', tickformat='+.0%')
fig.update_yaxes(title='Total positive EP (£)', tickprefix='£', tickformat=',.0f', rangemode='tozero')
fig.show()
pv = sens.pivot(index='change', columns='parameter', values='total_positive_ep')
pv = pv.loc[sorted(pv.index, key=lambda s: float(s.strip('%')))]
bind = sens.pivot(index='change', columns='parameter', values='budget_binds').loc[pv.index]
display(pv.round(0))
bind.rename(columns=lambda c: c + ' binds')

parameter,contact_cost,incentive,spread
change,,,
-50%,140070.0,124031.0,19046.0
-25%,120593.0,114004.0,57147.0
+0%,104414.0,104414.0,104414.0
+25%,92270.0,95116.0,158155.0
+50%,82020.0,86096.0,214674.0


parameter,contact_cost binds,incentive binds,spread binds
change,,,
-50%,False,True,False
-25%,False,True,False
+0%,True,True,True
+25%,True,True,True
+50%,True,False,True


The funding spread dominates. A 25% lower spread nearly halves the positive EP and the budget stops binding. Contact cost and incentive matter less. The campaign's value rests mainly on the funding-spread assumption, so it should be confirmed with Treasury before the results are used.

## 7. Save the campaign table

In [15]:
pl.save_campaign_table(t)
print(pl.CAMPAIGN_TABLE_PATH.name, t.shape)
t[['customer_id', 'segment', 'channel', 'eligible', 'propensity', 'deposit', 'value', 'contact_cost',
   'break_even_cost', 'expected_profit', 'ep_per_cost', 'rfm_code', 'engagement_score']].head()

campaign_table.parquet (10000, 53)


,customer_id,segment,channel,eligible,propensity,deposit,value,contact_cost,break_even_cost,expected_profit,ep_per_cost,rfm_code,engagement_score
0,C100010,Dormant,digital,1,0.044379,10593.0,140.3775,2.0,6.229868,4.229868,2.114934,113,0.0
1,C100011,Low engagement,digital,1,0.049981,10270.0,134.7250,2.0,6.733711,4.733711,2.366856,112,0.0
2,C100013,Low engagement,digital,1,0.027688,10006.0,130.1050,2.0,3.602376,1.602376,0.801188,111,0.0
3,C100017,Low engagement,digital,1,0.041916,10013.0,130.2275,2.0,5.458564,3.458564,1.729282,111,0.0
4,C100022,Dormant,digital,1,0.044379,10779.0,143.6325,2.0,6.374323,4.374323,2.187161,114,0.0


**Milestone 3 output:** `data/processed/campaign_table.parquet`. It has one row for each of the 10,000 customers, with eligibility, propensity, D, V, C, BE, EP, EP/C, both ranks, the segment, the RFM scores and the engagement score. This is the input to the genetic algorithm (M4).